In [3]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from transformers import AutoTokenizer, AutoModel

In [4]:
# =========================================================
# PATHS
# =========================================================
PROJECT_DIR = Path(r"C:\Users\Home\Desktop\projet-Ai")
SPLIT_DIR = PROJECT_DIR / "data" / "Split Dataset"
IMAGE_DIR = PROJECT_DIR / "data" / "Labelled Images"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = SPLIT_DIR / "Training_meme_dataset.csv"
VAL_CSV = SPLIT_DIR / "validation_meme_dataset.csv"
TEST_CSV = SPLIT_DIR / "Testing_meme_dataset.csv"

MODEL_PATH = MODEL_DIR / "meme-classification-multimodal_model_xpu-v2.pth"

In [5]:
# =========================================================
# SETTINGS
# =========================================================
IMAGE_COL = "image_name"
TEXT_COL = "sentence"
LABEL_COL = "label"

BATCH_SIZE = 8
NUM_EPOCHS = 12
NUM_WORKERS = 0
MAX_TEXT_LEN = 128

# choose models here
IMAGE_MODEL_NAME = "resnet50"   # options: resnet18, resnet50, efficientnet_b0
TEXT_MODEL_NAME = "bert-base-uncased"   # options: distilbert-base-uncased, bert-base-uncased, roberta-base

# learning rates
ENCODER_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4

# early stopping
PATIENCE = 3

In [6]:
# =========================================================
# DEVICE
# =========================================================
if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: xpu


In [7]:
# =========================================================
# LABEL CLEANING
# =========================================================
def clean_label(x):
    x = str(x).strip().lower()

    if x in ["non-offensiv", "non-offensive", "non offensive", "nonoffensive"]:
        return "non_offensive"
    if x in ["offensive", "offensiv"]:
        return "offensive"

    raise ValueError(f"Unexpected label found: {x}")


label_to_id = {
    "non_offensive": 0,
    "offensive": 1
}

id_to_label = {
    0: "non_offensive",
    1: "offensive"
}

In [8]:
# =========================================================
# LOAD CSV FILES
# =========================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

for df in [train_df, val_df, test_df]:
    df[LABEL_COL] = df[LABEL_COL].apply(clean_label)
    df[LABEL_COL] = df[LABEL_COL].map(label_to_id)
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label counts:")
print(train_df[LABEL_COL].value_counts().sort_index())

Train shape: (445, 3)
Val shape: (149, 3)
Test shape: (149, 3)

Train label counts:
label
0    258
1    187
Name: count, dtype: int64


In [9]:
# =========================================================
# CHECK IMAGE FILES
# =========================================================
def image_exists(image_name):
    image_path = IMAGE_DIR / str(image_name)
    return image_path.exists()

# verify images and optionally remove missing entries
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = df[~df[IMAGE_COL].apply(image_exists)]
    print(f"\nMissing images in {name}: {len(missing)}")
    if len(missing) > 0:
        # show a sample of the missing filenames
        print(missing[[IMAGE_COL]].head(10))
        # drop rows referencing missing files so downstream logic can continue
        df.drop(missing.index, inplace=True)
        print(f"Dropped {len(missing)} rows from {name} split")
        # you can also log or save `missing` if you wish to investigate further



Missing images in train: 0

Missing images in val: 1
    image_name
8  KbTk7Rq.png
Dropped 1 rows from val split

Missing images in test: 0


In [10]:
# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
# =========================================================
# TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)


# =========================================================
# DATASET
# =========================================================
class MemeMultimodalDataset(Dataset):
    def __init__(self, df, image_dir, tokenizer, max_text_len, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = str(self.df.loc[idx, IMAGE_COL])
        text = str(self.df.loc[idx, TEXT_COL])
        label = int(self.df.loc[idx, LABEL_COL])

        image_path = self.image_dir / image_name
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long)
        }

c:\Users\Home\Desktop\projet-Ai\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Home\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [12]:
# =========================================================
# DATALOADERS
# =========================================================
train_dataset = MemeMultimodalDataset(
    train_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, train_transform
)
val_dataset = MemeMultimodalDataset(
    val_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, eval_transform
)
test_dataset = MemeMultimodalDataset(
    test_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

In [13]:
# =========================================================
# MULTIMODAL MODEL
# =========================================================
class MultimodalMemeClassifier(nn.Module):
    def __init__(self, image_model_name, text_model_name, num_classes=2):
        super().__init__()

        # -------------------------
        # image encoder
        # -------------------------
        if image_model_name == "resnet18":
            self.image_encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            image_feature_dim = self.image_encoder.fc.in_features
            self.image_encoder.fc = nn.Identity()

        elif image_model_name == "resnet50":
            self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            image_feature_dim = self.image_encoder.fc.in_features
            self.image_encoder.fc = nn.Identity()

        elif image_model_name == "efficientnet_b0":
            self.image_encoder = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            image_feature_dim = self.image_encoder.classifier[1].in_features
            self.image_encoder.classifier = nn.Identity()

        else:
            raise ValueError(f"Unsupported image model: {image_model_name}")

        # -------------------------
        # text encoder
        # -------------------------
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_feature_dim = self.text_encoder.config.hidden_size

        # -------------------------
        # projection layers
        # -------------------------
        self.image_proj = nn.Sequential(
            nn.Linear(image_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # -------------------------
        # fusion + classifier
        # -------------------------
        self.fusion = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.LayerNorm(256)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        # image features
        image_features = self.image_encoder(image)
        image_features = self.image_proj(image_features)

        # text features
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # works for BERT / DistilBERT / RoBERTa
        text_features = text_outputs.last_hidden_state[:, 0, :]
        text_features = self.text_proj(text_features)

        # fusion
        fused = torch.cat([image_features, text_features], dim=1)
        fused = self.fusion(fused)

        logits = self.classifier(fused)
        return logits


model = MultimodalMemeClassifier(
    image_model_name=IMAGE_MODEL_NAME,
    text_model_name=TEXT_MODEL_NAME,
    num_classes=2
)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Home/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:42<00:00, 2.40MB/s]
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5039.71it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# =========================================================
# OPTIONAL: FREEZE ENCODERS FOR WARMUP
# =========================================================
FREEZE_IMAGE_ENCODER = False
FREEZE_TEXT_ENCODER = False

if FREEZE_IMAGE_ENCODER:
    for param in model.image_encoder.parameters():
        param.requires_grad = False

if FREEZE_TEXT_ENCODER:
    for param in model.text_encoder.parameters():
        param.requires_grad = False

In [15]:
# =========================================================
# LOSS / OPTIMIZER
# =========================================================
class_counts = train_df[LABEL_COL].value_counts().sort_index().values
class_weights = torch.tensor(
    [len(train_df) / (2 * class_counts[0]), len(train_df) / (2 * class_counts[1])],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW([
    {"params": model.image_encoder.parameters(), "lr": ENCODER_LR},
    {"params": model.text_encoder.parameters(), "lr": ENCODER_LR},
    {"params": model.image_proj.parameters(), "lr": HEAD_LR},
    {"params": model.text_proj.parameters(), "lr": HEAD_LR},
    {"params": model.fusion.parameters(), "lr": HEAD_LR},
    {"params": model.classifier.parameters(), "lr": HEAD_LR},
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

use_amp = (device.type == "xpu")

In [16]:
# =========================================================
# TRAIN FUNCTION
# =========================================================
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc

In [17]:
# =========================================================
# EVAL FUNCTION
# =========================================================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_preds = []

    for batch in loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [18]:
# =========================================================
# TRAIN LOOP
# =========================================================
MODEL_PATH = MODEL_DIR / f"best_{IMAGE_MODEL_NAME}_{TEXT_MODEL_NAME.replace('/', '_')}.pth"

best_val_f1 = 0.0
early_stop_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        early_stop_counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label_to_id": label_to_id,
                "id_to_label": id_to_label,
                "image_model_name": IMAGE_MODEL_NAME,
                "text_model_name": TEXT_MODEL_NAME
            },
            MODEL_PATH
        )
        print("Best model saved.")
    else:
        early_stop_counter += 1
        print(f"No improvement. Early stop counter: {early_stop_counter}/{PATIENCE}")

    scheduler.step(val_f1)

    if early_stop_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining finished.")
print("Best validation F1:", best_val_f1)


Epoch 1/12
Train Loss: 0.7117 | Train Acc: 0.5191
Val   Loss: 0.6948 | Val   Acc: 0.4932 | Val F1: 0.4927
Best model saved.

Epoch 2/12
Train Loss: 0.7028 | Train Acc: 0.5236
Val   Loss: 0.6847 | Val   Acc: 0.5608 | Val F1: 0.5549
Best model saved.

Epoch 3/12
Train Loss: 0.6719 | Train Acc: 0.5798
Val   Loss: 0.6854 | Val   Acc: 0.5608 | Val F1: 0.5608
Best model saved.

Epoch 4/12
Train Loss: 0.6151 | Train Acc: 0.6629
Val   Loss: 0.7084 | Val   Acc: 0.5473 | Val F1: 0.5473
No improvement. Early stop counter: 1/3

Epoch 5/12
Train Loss: 0.4801 | Train Acc: 0.8180
Val   Loss: 0.8035 | Val   Acc: 0.5811 | Val F1: 0.5783
Best model saved.

Epoch 6/12
Train Loss: 0.3350 | Train Acc: 0.8854
Val   Loss: 1.1556 | Val   Acc: 0.5270 | Val F1: 0.5262
No improvement. Early stop counter: 1/3

Epoch 7/12
Train Loss: 0.2015 | Train Acc: 0.9528
Val   Loss: 1.2552 | Val   Acc: 0.6149 | Val F1: 0.5920
Best model saved.

Epoch 8/12
Train Loss: 0.0863 | Train Acc: 0.9865
Val   Loss: 1.5316 | Val   Acc

In [19]:
# =========================================================
# TEST THE TRAINED MULTIMODAL MODEL
# =========================================================
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc, test_f1, test_labels, test_preds = evaluate(model, test_loader)

print("\nTest results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")
print(f"Test F1  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=["non_offensive", "offensive"],
    digits=4
))


Test results
Test Loss: 1.7882
Test Acc : 0.5772
Test F1  : 0.5407

Confusion Matrix:
[[64 27]
 [36 22]]

Classification Report:
               precision    recall  f1-score   support

non_offensive     0.6400    0.7033    0.6702        91
    offensive     0.4490    0.3793    0.4112        58

     accuracy                         0.5772       149
    macro avg     0.5445    0.5413    0.5407       149
 weighted avg     0.5656    0.5772    0.5694       149



In [20]:
# =========================================================
# RUN MULTIPLE MODEL COMBINATIONS
# =========================================================
def run_experiment(image_model_name, text_model_name):
    print("=" * 80)
    print(f"Running experiment: {image_model_name} + {text_model_name}")
    print("=" * 80)

    model = MultimodalMemeClassifier(
        image_model_name=image_model_name,
        text_model_name=text_model_name,
        num_classes=2
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.AdamW([
        {"params": model.image_encoder.parameters(), "lr": ENCODER_LR},
        {"params": model.text_encoder.parameters(), "lr": ENCODER_LR},
        {"params": model.image_proj.parameters(), "lr": HEAD_LR},
        {"params": model.text_proj.parameters(), "lr": HEAD_LR},
        {"params": model.fusion.parameters(), "lr": HEAD_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )

    best_val_f1 = 0.0
    early_stop_counter = 0

    best_model_path = MODEL_DIR / f"best_{image_model_name}_{text_model_name.replace('/', '_')}.pth"

    def train_one_epoch_local(model, loader):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for batch in loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            if use_amp:
                with torch.autocast(device_type="xpu", dtype=torch.float16):
                    outputs = model(images, input_ids, attention_mask)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

        return total_loss / total_samples, total_correct / total_samples

    @torch.no_grad()
    def evaluate_local(model, loader):
        model.eval()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0
        all_labels = []
        all_preds = []

        for batch in loader:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            if use_amp:
                with torch.autocast(device_type="xpu", dtype=torch.float16):
                    outputs = model(images, input_ids, attention_mask)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

        epoch_loss = total_loss / total_samples
        epoch_acc = total_correct / total_samples
        epoch_f1 = f1_score(all_labels, all_preds, average="macro")

        return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch_local(model, train_loader)
        val_loss, val_acc, val_f1, _, _ = evaluate_local(model, val_loader)

        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            early_stop_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print("Best model saved.")
        else:
            early_stop_counter += 1

        scheduler.step(val_f1)

        if early_stop_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    test_loss, test_acc, test_f1, _, _ = evaluate_local(model, test_loader)

    print("\nFinal Test Results")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Acc : {test_acc:.4f}")
    print(f"Test F1  : {test_f1:.4f}")

    return {
        "image_model": image_model_name,
        "text_model": text_model_name,
        "best_val_f1": best_val_f1,
        "test_acc": test_acc,
        "test_f1": test_f1
    }

In [21]:
# =========================================================
# EXPERIMENT LIST
# =========================================================
experiments = [
    ("resnet18", "distilbert-base-uncased"),
    ("resnet50", "distilbert-base-uncased"),
    ("resnet50", "bert-base-uncased"),
    ("efficientnet_b0", "bert-base-uncased"),
    ("efficientnet_b0", "roberta-base"),
]

In [22]:
results = []

for image_model_name, text_model_name in experiments:
    result = run_experiment(image_model_name, text_model_name)
    results.append(result)

Running experiment: resnet18 + distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13901.77it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/12
Train Loss: 0.7093 | Train Acc: 0.4831
Val   Loss: 0.6810 | Val   Acc: 0.5878 | Val F1: 0.5100
Best model saved.

Epoch 2/12
Train Loss: 0.6982 | Train Acc: 0.5213
Val   Loss: 0.6846 | Val   Acc: 0.6081 | Val F1: 0.4219

Epoch 3/12
Train Loss: 0.6984 | Train Acc: 0.5101
Val   Loss: 0.6807 | Val   Acc: 0.6149 | Val F1: 0.5079

Epoch 4/12
Train Loss: 0.6796 | Train Acc: 0.5461
Val   Loss: 0.6777 | Val   Acc: 0.5743 | Val F1: 0.5219
Best model saved.

Epoch 5/12
Train Loss: 0.6752 | Train Acc: 0.5685
Val   Loss: 0.6690 | Val   Acc: 0.5676 | Val F1: 0.5213

Epoch 6/12
Train Loss: 0.6424 | Train Acc: 0.6629
Val   Loss: 0.6780 | Val   Acc: 0.5811 | Val F1: 0.5363
Best model saved.

Epoch 7/12
Train Loss: 0.5585 | Train Acc: 0.7551
Val   Loss: 0.8099 | Val   Acc: 0.5203 | Val F1: 0.5122

Epoch 8/12
Train Loss: 0.4272 | Train Acc: 0.8337
Val   Loss: 0.9790 | Val   Acc: 0.5338 | Val F1: 0.5276

Epoch 9/12
Train Loss: 0.3342 | Train Acc: 0.9034
Val   Loss: 1.1810 | Val   Acc: 0.5473 

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8217.52it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/12
Train Loss: 0.7096 | Train Acc: 0.4809
Val   Loss: 0.6918 | Val   Acc: 0.4865 | Val F1: 0.4804
Best model saved.

Epoch 2/12
Train Loss: 0.6930 | Train Acc: 0.5348
Val   Loss: 0.6856 | Val   Acc: 0.5541 | Val F1: 0.5458
Best model saved.

Epoch 3/12
Train Loss: 0.6827 | Train Acc: 0.5865
Val   Loss: 0.7172 | Val   Acc: 0.4257 | Val F1: 0.3672

Epoch 4/12
Train Loss: 0.6656 | Train Acc: 0.6022
Val   Loss: 0.6866 | Val   Acc: 0.5473 | Val F1: 0.5468
Best model saved.

Epoch 5/12
Train Loss: 0.5834 | Train Acc: 0.7169
Val   Loss: 0.7898 | Val   Acc: 0.5270 | Val F1: 0.5262

Epoch 6/12
Train Loss: 0.4854 | Train Acc: 0.7865
Val   Loss: 0.8846 | Val   Acc: 0.5338 | Val F1: 0.5327

Epoch 7/12
Train Loss: 0.2847 | Train Acc: 0.9056
Val   Loss: 1.1403 | Val   Acc: 0.6216 | Val F1: 0.5918
Best model saved.

Epoch 8/12
Train Loss: 0.1445 | Train Acc: 0.9685
Val   Loss: 1.3984 | Val   Acc: 0.6351 | Val F1: 0.5961
Best model saved.

Epoch 9/12
Train Loss: 0.0995 | Train Acc: 0.9775
Val

...


Epoch 1/12
Train Loss: 0.7079 | Train Acc: 0.5056
Val   Loss: 0.6980 | Val   Acc: 0.4865 | Val F1: 0.4674
Best model saved.

Epoch 2/12
Train Loss: 0.6908 | Train Acc: 0.5663
Val   Loss: 0.7265 | Val   Acc: 0.3851 | Val F1: 0.2866

Epoch 3/12
Train Loss: 0.6819 | Train Acc: 0.5843
Val   Loss: 0.7082 | Val   Acc: 0.4459 | Val F1: 0.4310

Epoch 4/12
Train Loss: 0.6486 | Train Acc: 0.6427
Val   Loss: 0.7065 | Val   Acc: 0.5743 | Val F1: 0.5389
Best model saved.

Epoch 5/12
Train Loss: 0.6260 | Train Acc: 0.6764
Val   Loss: 0.7129 | Val   Acc: 0.5608 | Val F1: 0.5348

Epoch 6/12
Train Loss: 0.5402 | Train Acc: 0.7753
Val   Loss: 0.7774 | Val   Acc: 0.6014 | Val F1: 0.5876
Best model saved.

Epoch 7/12
Train Loss: 0.4448 | Train Acc: 0.8292
Val   Loss: 0.8913 | Val   Acc: 0.5946 | Val F1: 0.5919
Best model saved.

Epoch 8/12
Train Loss: 0.3065 | Train Acc: 0.9079
Val   Loss: 1.2003 | Val   Acc: 0.5811 | Val F1: 0.5363

Epoch 9/12
Train Loss: 0.1949 | Train Acc: 0.9528
Val   Loss: 1.3524 | 

100%|██████████| 20.5M/20.5M [00:09<00:00, 2.37MB/s]
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10234.90it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/12
Train Loss: 0.7292 | Train Acc: 0.4876
Val   Loss: 0.6889 | Val   Acc: 0.5405 | Val F1: 0.5116
Best model saved.

Epoch 2/12
Train Loss: 0.6933 | Train Acc: 0.5551
Val   Loss: 0.6842 | Val   Acc: 0.5270 | Val F1: 0.5267
Best model saved.

Epoch 3/12
Train Loss: 0.6858 | Train Acc: 0.5528
Val   Loss: 0.7250 | Val   Acc: 0.3986 | Val F1: 0.3246

Epoch 4/12
Train Loss: 0.6522 | Train Acc: 0.6404
Val   Loss: 0.7457 | Val   Acc: 0.5946 | Val F1: 0.4150

Epoch 5/12
Train Loss: 0.5543 | Train Acc: 0.7551
Val   Loss: 0.7910 | Val   Acc: 0.5946 | Val F1: 0.5934
Best model saved.

Epoch 6/12
Train Loss: 0.4214 | Train Acc: 0.8449
Val   Loss: 0.9491 | Val   Acc: 0.5608 | Val F1: 0.5203

Epoch 7/12
Train Loss: 0.3089 | Train Acc: 0.9079
Val   Loss: 1.1084 | Val   Acc: 0.5676 | Val F1: 0.5625

Epoch 8/12
Train Loss: 0.1707 | Train Acc: 0.9618
Val   Loss: 1.3057 | Val   Acc: 0.5743 | Val F1: 0.5389
Early stopping triggered.

Final Test Results
Test Loss: 0.7174
Test Acc : 0.5839
Test F1 

c:\Users\Home\Desktop\projet-Ai\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Home\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9554.00it/s]
RobertaModel LOAD REPORT from: roberta-base
Key     


Epoch 1/12
Train Loss: 0.7175 | Train Acc: 0.4809
Val   Loss: 0.7183 | Val   Acc: 0.6149 | Val F1: 0.3808
Best model saved.

Epoch 2/12
Train Loss: 0.7173 | Train Acc: 0.5056
Val   Loss: 0.6880 | Val   Acc: 0.6149 | Val F1: 0.3808

Epoch 3/12
Train Loss: 0.7144 | Train Acc: 0.5236
Val   Loss: 0.6869 | Val   Acc: 0.5811 | Val F1: 0.5363
Best model saved.

Epoch 4/12
Train Loss: 0.6935 | Train Acc: 0.5573
Val   Loss: 0.6787 | Val   Acc: 0.5878 | Val F1: 0.4816

Epoch 5/12
Train Loss: 0.6881 | Train Acc: 0.5708
Val   Loss: 0.6741 | Val   Acc: 0.6216 | Val F1: 0.5584
Best model saved.

Epoch 6/12
Train Loss: 0.6908 | Train Acc: 0.5708
Val   Loss: 0.6784 | Val   Acc: 0.5743 | Val F1: 0.5686
Best model saved.

Epoch 7/12
Train Loss: 0.6756 | Train Acc: 0.5798
Val   Loss: 0.6724 | Val   Acc: 0.6216 | Val F1: 0.4418

Epoch 8/12
Train Loss: 0.6935 | Train Acc: 0.5551
Val   Loss: 0.6732 | Val   Acc: 0.5946 | Val F1: 0.4689

Epoch 9/12
Train Loss: 0.6770 | Train Acc: 0.6157
Val   Loss: 0.6688 | 

In [23]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="test_f1", ascending=False).reset_index(drop=True)
results_df

,image_model,text_model,best_val_f1,test_acc,test_f1
0,resnet18,distilbert-base-uncased,0.536285,0.697987,0.659662
1,resnet50,bert-base-uncased,0.591912,0.617450,0.615719
2,resnet50,distilbert-base-uncased,0.596119,0.644295,0.599157
3,efficientnet_b0,bert-base-uncased,0.593407,0.583893,0.582972
4,efficientnet_b0,roberta-base,0.625460,0.530201,0.516593
